In [1]:
from pyspark.sql import SparkSession

In [2]:
import os

In [3]:
import os
import subprocess

from pyspark.sql import SparkSession


# Lấy IP hiện tại của Ubuntu WSL.
# Ví dụ máy bạn hiện tại trả về: 172.22.188.14
WSL_IP = subprocess.check_output(
    ["hostname", "-I"],
    text=True,
).split()[0]


# Tách endpoint MinIO ra thành biến để sau này dễ thay đổi.
MINIO_ENDPOINT = os.getenv(
    "MINIO_ENDPOINT",
    "http://172.22.176.1:9000",
)


spark = (
    SparkSession.builder
    .appName("CareerSignal")

    # Driver trong WSL kết nối tới cổng 7077
    # mà Docker đã publish ra host.
    .master("spark://localhost:7077")

    # ---------------------------------------------------------
    # CẤU HÌNH MỚI: KẾT NỐI WORKER/EXECUTOR VỀ DRIVER TRONG WSL
    # ---------------------------------------------------------

    # Địa chỉ mà Driver công bố cho Master và Executor.
    # Executor trong Docker sẽ kết nối tới IP WSL này.
    .config(
        "spark.driver.host",
        WSL_IP,
    )

    # Driver lắng nghe kết nối trên tất cả interface mạng của WSL.
    .config(
        "spark.driver.bindAddress",
        "0.0.0.0",
    )

    # Cố định cổng RPC của Driver.
    .config(
        "spark.driver.port",
        "39001",
    )

    # Cố định cổng BlockManager riêng của Driver.
    .config(
        "spark.driver.blockManager.port",
        "39002",
    )

    # ---------------------------------------------------------
    # THƯ VIỆN ĐỂ SPARK ĐỌC S3/MINIO
    # ---------------------------------------------------------

    .config(
        "spark.jars.packages",
        (
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262"
        ),
    )

    # ---------------------------------------------------------
    # CẤU HÌNH MINIO
    # ---------------------------------------------------------

    .config(
        "spark.hadoop.fs.s3a.endpoint",
        MINIO_ENDPOINT,
    )

    .config(
        "spark.hadoop.fs.s3a.access.key",
        os.environ["MINIO_USER"],
    )

    .config(
        "spark.hadoop.fs.s3a.secret.key",
        os.environ["MINIO_PASSWORD"],
    )

    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )

    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true",
    )

    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false",
    )

    .config(
        "spark.hadoop.fs.s3a.endpoint.region",
        "us-east-1",
    )

    .getOrCreate()
)

26/08/14 13:51:23 WARN Utils: Your hostname, DESKTOP-HHFRTS1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/14 13:51:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/legion/.virtualenvs/careersignal-py310/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/legion/.ivy2/cache
The jars for the packages stored in: /home/legion/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8cc80bb6-ca2d-4923-afb7-74c89971d7ba;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 202ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evic

In [4]:
data=spark.read.parquet("s3a://bronze/2026-7-30/1785403101.108542-ITViec.parquet")

26/08/14 13:51:29 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [5]:
data_ITViec=spark.read.parquet("s3a://silver/ITViec/2026-08-07/")

In [6]:
data_ITViec.show()

+--------------------+----------+----------+--------+------+------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+-------------------+--------------------+
|              job_id|min_salary|max_salary|currency|period|parse_status|               title|             job_url|        company_name|working_model|            location|              skills|            benefits|          posted_at|          scraped_at|
+--------------------+----------+----------+--------+------+------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+-------------------+--------------------+
|49e15baa-c9f7-473...|      NULL|      NULL|    NULL|  NULL|     success|DevOps Expert - I...|https://itviec.co...|              VPBank|    At office|              Ha Noi|[DevOps, Azure, A...|                  []| Posted\n7 days ago|20

In [16]:
dataTOPCV=spark.read.parquet("s3a://silver/topcv/2026-08-05")

In [17]:
dataTOPCV.show()

+-------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+----------+----------+---------------+-------------+------------------+-------------------+--------------------+--------------------+--------------------+---------+----------------+-----------------+-------------+------------+-----------+
| job_id|             job_url|               title|                city|        company_name|           apply_url|verification_level|salary_min|salary_max|salary_currency|salary_period|salary_parse_error|         scraped_at|          role_group|        primary_role|     secondary_roles|seniority|experience_years|           skills|is_multi_role|parse_status|parse_error|
+-------+--------------------+--------------------+--------------------+--------------------+--------------------+------------------+----------+----------+---------------+-------------+------------------+-------------------+--------------------+-----------

data.select("skills")

In [18]:
spark.stop()

In [7]:
from pyspark.sql import functions as F
data.filter(
    F.col("skills").isNotNull()
).select("skills").show(truncate=False)

+----------------------------------------------------------------------------------------------------------------+
|skills                                                                                                          |
+----------------------------------------------------------------------------------------------------------------+
|[Cybersecurity, Cloud Security, DevOps, Team Management, English]                                               |
|[Automation Test, Prompt Engineering, English, Tester]                                                          |
|[Project Management, Japanese IT Communication, Lean Project Management, Japanese, Software Architecture, Agile]|
|[Project Management, Scrum, Agile, .NET, Japanese, Bridge Engineer]                                             |
|[IT Audit, Risk Management, IT Governance, ISO 27001, VBA, SQL]                                                 |
|[Java, MySQL, OOP, PostgreSql, Spring Boot, Docker]                            

In [8]:
data.createOrReplaceTempView("jobs")

In [9]:
spark.sql("""
    Select salary,count(salary) as cac_loai_salary
    from jobs
    GROUP BY(salary)
    order by cac_loai_salary desc
""").show(truncate=False)

+------------------------+---------------+
|salary                  |cac_loai_salary|
+------------------------+---------------+
|You'll love it          |606            |
|Very attractive!!!      |18             |
|800 - 1,500 USD         |13             |
|1,000 - 2,000 USD       |11             |
|2,000 - 3,000 USD       |9              |
|800 - 1,200 USD         |7              |
|1,000 - 1,500 USD       |7              |
|1,500 - 2,500 USD       |7              |
|2,000 - 2,500 USD       |6              |
|700 - 1,200 USD         |4              |
|1,500 - 2,000 USD       |4              |
|1,000 - 3,000 USD       |4              |
|800 - 1,000 USD         |4              |
|800 - 2,000 USD         |4              |
|1,200 - 2,000 USD       |3              |
|1,000 - 1,600 USD       |3              |
|1,000 - 2,500 USD       |3              |
|Thỏa thuận theo năng lực|3              |
|200 - 500 USD           |3              |
|2,000 - 5,000 USD       |3              |
+----------

In [10]:
spark.sql("""
    select slug
    from jobs


""").show(truncate=False)

+-------------------------------------------------------------------------------------+
|slug                                                                                 |
+-------------------------------------------------------------------------------------+
|lead-penetration-tester-pentester-itviec-recruitment-consulting-3432                 |
|ai-qa-data-quality-specialist-hybrid-qa-money-forward-vietnam-co-ltd-5927            |
|senior-manager-japanese-speaking-itviec-recruitment-consulting-3643                  |
|project-leader-project-manager-japanese-n2-itviec-recruitment-consulting-3806        |
|auditor-it-model-data-co-quan-kiem-toan-noi-bo-mb-bank-3602                          |
|middle-senior-java-developer-ngan-hang-tmcp-phuong-dong-ocb-0831                     |
|senior-android-engineer-grab-vietnam-ltd-5050                                        |
|sr-net-backend-developer-asp-net-c-sql-reactjs-grapecity-3106                        |
|android-developer-kotlin-up-to-

In [11]:
spark.sql("""
    select title
    from jobs


""").show(truncate=False)

+------------------------------------------------------+
|title                                                 |
+------------------------------------------------------+
|Lead Penetration Tester (Pentester)                   |
|AI QA & Data Quality Specialist (Hybrid QA)           |
|Senior Manager (Japanese Speaking)                    |
|Project Leader/ Project Manager (Japanese N2+)        |
|Internal Auditor/ Senior Internal Auditor (IT/Data)   |
|Middle/ Senior Java Developer                         |
|Senior Android Engineer                               |
|Sr .NET Backend Developer (ASP.NET, C#, SQL, ReactJS) |
|Android Developer (Kotlin) Up To 45M                  |
|An ninh thông tin (Security Engineer) - TNTech - 2V068|
|Expert Information Security (HN&HCM)                  |
|Product Owner                                         |
|Senior Automation Test (AI, QA QC, API)               |
|Project Manager                                       |
|Embedded Software Engineer (MC

In [12]:
spark.sql("""
    select *
    from jobs


""").show(truncate=False)

+------------------------------------+-------------------------------------------------------------------------------------+------------------------------------------------------+----------------------------------------------------------------------------------------------------------------+------------------------------------------+---------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+--------------------------------+-------------+--------------------+----------------------------------------------------------------------------------------------------------------+---------------------

In [13]:
data.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+---------+--------------------+-----------+--------------------+
|              job_id|                slug|               title|             job_url|        company_name|         company_url|        company_logo|              salary|        job_category|working_model|            location|              skills|            benefits|    label|           posted_at|source_page|          scraped_at|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+---------+--------------------+-----------+--------------------+
|38c

In [14]:
data_salary=data.select("job_id","salary")

In [15]:
data_new=data_salary.repartition(8).rdd.mapPartitions(parse_partition)

NameError: name 'parse_partition' is not defined

In [ ]:
parsed_schema = StructType([
    StructField("job_id", StringType(), False),
    StructField("min_salary", DoubleType(), True),
    StructField("max_salary", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("period", StringType(), True),
    StructField("parse_status", StringType(), False),
    ])

In [ ]:
dataframe=spark.createDataFrame(
    data_new,
    parsed_schema
)

In [ ]:
dataframe.show()

+--------------------+----------+----------+--------+------+-------------+
|              job_id|min_salary|max_salary|currency|peroid|parsed_status|
+--------------------+----------+----------+--------+------+-------------+
|6f35d25f-80fb-40c...|    2500.0|    3500.0|     USD| month|      success|
|7ed02753-b3a9-4b2...|      NULL|      NULL|     VND| month|      success|
|5018ebfc-3576-440...|      NULL|      NULL|     VND| month|      success|
|37c93be2-8639-469...|    1000.0|    1600.0|     USD| month|      success|
|b5c079ee-b9e0-484...|    1000.0|    3000.0|     USD| month|      success|
|f8076deb-338e-498...|     700.0|    1000.0|     USD| month|      success|
|ad14c281-34e5-47e...|      NULL|      NULL|     VND| month|      success|
|42f64c20-f4da-483...|      NULL|      NULL|     VND| month|      success|
|340582b8-1f73-406...|      NULL|      NULL|     VND| month|      success|
|7972b13f-a6d9-41c...|    2000.0|    3000.0|     USD| month|      success|
|81f3f740-2272-4d7...|   

26/08/07 18:29:39 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 528343 ms exceeds timeout 120000 ms
26/08/07 18:29:39 ERROR TaskSchedulerImpl: Lost executor 0 on 172.19.0.4: Executor heartbeat timed out after 528343 ms


In [ ]:
data.show()

+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+---------+--------------------+-----------+--------------------+
|              job_id|                slug|               title|             job_url|        company_name|         company_url|        company_logo|              salary|        job_category|working_model|            location|              skills|            benefits|    label|           posted_at|source_page|          scraped_at|
+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+---------+--------------------+-----------+--------------------+
|38c

In [ ]:
data.columns

['job_id',
 'slug',
 'title',
 'job_url',
 'company_name',
 'company_url',
 'company_logo',
 'salary',
 'job_category',
 'working_model',
 'location',
 'skills',
 'benefits',
 'label',
 'posted_at',
 'source_page',
 'scraped_at']

In [ ]:
data_to_join=data.select(
    "job_id",
    "title",
    "job_url",
    "company_name",
    "working_model",
    "location",
    "skills",
    "benefits",
    "posted_at",
    "scraped_at"
)

In [ ]:
data_final=dataframe.join(
    data_to_join,
    on="job_id",
    how="inner"
)

In [ ]:
data_final.show()

+--------------------+----------+----------+--------+------+-------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+-------------------+--------------------+
|              job_id|min_salary|max_salary|currency|peroid|parsed_status|               title|             job_url|        company_name|working_model|            location|              skills|            benefits|          posted_at|          scraped_at|
+--------------------+----------+----------+--------+------+-------------+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+-------------------+--------------------+
|6f35d25f-80fb-40c...|    2500.0|    3500.0|     USD| month|      success|[HCM] Bridge Engi...|https://itviec.co...| Hybrid Technologies|    At office|         Ho Chi Minh|[Bridge Engineer,...|[Nâng cao kĩ thuậ...| Posted\n9 days ag

In [ ]:
spark.read.parquet("s3://silver/ITViec/2026-08-07")

26/08/13 20:58:18 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3://silver/ITViec/2026-08-07.
org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3443)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.streaming.FileStreamSink$.hasMetadata(FileStreamSink.scala:53)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:366)
	at org.apache.spark.sql.DataFrameReader.loadV1Source(DataFrameReader.scala:229)
	a

Py4JJavaError: An error occurred while calling o86.parquet.
: org.apache.hadoop.fs.UnsupportedFileSystemException: No FileSystem for scheme "s3"
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3443)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.java:3466)
	at org.apache.hadoop.fs.FileSystem.access$300(FileSystem.java:174)
	at org.apache.hadoop.fs.FileSystem$Cache.getInternal(FileSystem.java:3574)
	at org.apache.hadoop.fs.FileSystem$Cache.get(FileSystem.java:3521)
	at org.apache.hadoop.fs.FileSystem.get(FileSystem.java:540)
	at org.apache.hadoop.fs.Path.getFileSystem(Path.java:365)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:724)
	at scala.collection.immutable.List.map(List.scala:293)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:722)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(